# Cleaning data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/LLM_Data_2 - Grad_Project_Data.csv")
df.head()

,Question,Answer
0,Did ancient Egyptian women have a high social ...,"Yes, they enjoyed a relatively high social sta..."
1,How did property inheritance work in ancient E...,All landed property was passed down through th...
2,Why was property passed down through the femal...,It was based on the assumption that maternity ...
3,Did ancient Egyptian women have to wear veils ...,"No, unlike the women of ancient Greece, they e..."
4,How were male and female guests seated at form...,Married guests sat together in pairs on fine c...


In [ ]:
def format_row(row):
    q = str(row["Question"]).strip()
    a = str(row["Answer"]).strip()

    return f"Question: {q} Answer: {a}"

df["text"] = df.apply(format_row, axis=1)

In [ ]:
df = df[df["text"].str.len() > 20]
df = df.drop_duplicates(subset=["text"])

In [ ]:
df[["text"]].to_csv("cleaned_data.csv", index=False)

In [ ]:
df["text"].iloc[0]

'Question: Did ancient Egyptian women have a high social status? Answer: Yes, they enjoyed a relatively high social status and could exert influence outside their domestic roles.'

# Arabic qwen

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.2 MB/s eta 0:00:00


In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])
    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    return results[:k]

In [ ]:
def answer_question(query):
    context_list = retrieve(query)
    context = "\n".join(context_list)

    prompt = f"""
    You are an expert Egyptologist.

    Answer the question using the context.

    Rules:
    - Combine relevant information into a complete answer.
    - Answer in 2–3 sentences.
    - Do NOT focus on only one detail.
    - Do NOT add information outside the context.

    Context:
    {context}

    Question: {query}
    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Answer:")[-1].strip()

    if not answer.endswith((".", "!", "?")):
        if "." in answer:
            answer = answer.rsplit(".", 1)[0] + "."

    return answer

In [ ]:
from deep_translator import GoogleTranslator

def is_arabic(text):
    return any('\u0600' <= c <= '\u06FF' for c in text)

def answer_question_multilang(query):

    arabic = is_arabic(query)

    try:
        if arabic:
            query_en = GoogleTranslator(source='auto', target='en').translate(query)
        else:
            query_en = query
    except:
        return "حدث خطأ في الترجمة" if arabic else "Translation error"


    answer_en = answer_question(query_en)


    try:
        if arabic:
            answer_ar = GoogleTranslator(source='auto', target='ar').translate(answer_en)
            return answer_ar
    except:
        return answer_en

    return answer_en

In [ ]:
answer_question_multilang("مين كانت حتشبسوت؟")

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


'كانت حتشبسوت ابنة زوجة وأخت زوجها (حيث تزوجت من أخيها غير الشقيق) للفرعون تحتمس الثاني، وخدمت في البداية تحت قيادته قبل أن تصبح وصية مشتركة مع ابن أخيها وابنها المتبنى تحتمس الثالث بعد أن بلغ سن الرشد. وفي وقت لاحق، حكمت بمفردها كفرعون لما يقرب من عقدين من الزمن حتى خلعها تحتمس الثالث خلال حياته اللاحقة.'

In [ ]:
answer_question_multilang("امتى حكمت حتشبسوت؟")

'حكمت حتشبسوت من عام 1504 قبل الميلاد تقريبًا حتى عام 1490 قبل الميلاد تقريبًا، وهي فترة حكمها التي أعقبها حكمها المستقل الكامل الذي استمر لحوالي 21 عامًا أو نحو ذلك اعتمادًا على مصادر وتفسيرات مختلفة. على وجه الدقة، حكمت بين ج. 1507 – 1458 قبل الميلاد بناءً على بعض التقديرات العلمية. يمكن أن تختلف المدة المحددة قليلاً ولكنها تقع بشكل عام ضمن هذا النطاق.'

In [ ]:
answer_question_multilang("مين هو رمسيس الثاني؟")

'رمسيس الثاني (مكتوب أيضًا رمسيس أو رمسيس) كان الفرعون الثالث من الأسرة التاسعة عشرة في مصر وحكم من عام 1279 تقريبًا حتى وفاته حوالي عام 1213 قبل الميلاد. ويعتبر أحد أبرز حكام مصر القديمة بسبب فترة حكمه الطويلة ومشاريع البناء العديدة في مواقع مختلفة في جميع أنحاء مصر.'